In [1]:
!git clone https://github.com/sinh2206/O_D.git

Cloning into 'O_D'...
remote: Enumerating objects: 9081, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 9081 (delta 21), reused 37 (delta 11), pack-reused 9031 (from 2)
Receiving objects: 100% (9081/9081), 996.84 MiB | 50.58 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Updating files: 100% (9020/9020), done.


In [2]:
%cd O_D
!ls

/content/O_D
LICENSE  object_detection.ipynb  public     requirements.txt  train.py
models	 predict.py		 README.md  results	      utils


In [3]:
%pip install -r requirements.txt

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 154MB/s] 
Device: cuda:0 (Tesla T4, 14.6 GB), AMP: True
DataLoader workers: 2 (max safe: 2), pin_memory: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Epoch 001/040 | train_loss=2.5839 (obj=0.0466, noobj=0.9956, box=0.2526, cls=0.5902, iou=0.5283) | val_loss=1.9191
Saved best checkpoint: models/best.pth (val_loss=1.9191)
Epoch 002/040 | train_loss=1.9362 (obj=0.0345, noobj=0.7613, box=0.1969, cls=0.3986, iou=0.5599) | val_loss=1.9214
Epoch 003/040 | train_loss=1.7895 (obj=0.0337, noobj=0.7265, box=0.1797, cls=0.3597, iou=0.5782) | val_loss=1.8616
Saved best checkpoint: models/best.pth (val_loss=1.8616)
Epoch 004/040 | train_loss=1.6459 (obj=0.0298, noobj=0.6436, box=0.1645, cls=0.3525, iou=0.5930) | val_loss=2.0210
Epoch 005/040 | train_loss=1.5155 (obj=0.0276, noobj=0.6181, b

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda:0 (Tesla T4, 14.6 GB)
Predicted images: 1500
Saved JSON: val_predictions.json
Saved preview images: 50 -> results


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.260478,
  "performance_points": 10,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 62946,
  "micro_precision": 0.02186,
  "micro_recall": 0.680851,
  "per_class": {
    "person": {
      "ap": 0.34405,
      "num_ground_truth": 1074,
      "num_predictions": 15431,
      "true_positives": 744,
      "false_positives": 14687,
      "recall": 0.692737,
      "precision": 0.048215
    },
    "car": {
      "ap": 0.272153,
      "num_ground_truth": 283,
      "num_predictions": 5674,
      "true_positives": 137,
      "false_positives": 5537,
      "recall": 0.484099,
      "precision": 0.024145
    },
    "dog": {
      "ap": 0.215987,
      "num_ground_truth": 206,
      "num_predictions": 5001,
      "true_positives": 149,
      "false_positives": 4852,
      "recall": 0.723301,
      "precision": 0.029794
    },
    "cat": {
      "ap": 0.405688,
      "num_ground_truth": 176,
      "num_predictions": 4048,
      "true_positives": 143,
 

In [7]:
!python predict.py \
  --image_dir ./public/train/images \
  --output train_predictions.json \
  --model_path ./models/best.pth

Device: cuda:0 (Tesla T4, 14.6 GB)
Predicted images: 7500
Saved JSON: train_predictions.json
Saved preview images: 50 -> results


In [8]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/train.json \
  --predictions train_predictions.json \
  --output train_score.json

{
  "mAP@0.5": 0.308932,
  "performance_points": 10,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 10642,
  "num_predictions": 308689,
  "micro_precision": 0.02531,
  "micro_recall": 0.734167,
  "per_class": {
    "person": {
      "ap": 0.350976,
      "num_ground_truth": 5829,
      "num_predictions": 79295,
      "true_positives": 4246,
      "false_positives": 75049,
      "recall": 0.728427,
      "precision": 0.053547
    },
    "car": {
      "ap": 0.324726,
      "num_ground_truth": 1339,
      "num_predictions": 27551,
      "true_positives": 804,
      "false_positives": 26747,
      "recall": 0.600448,
      "precision": 0.029182
    },
    "dog": {
      "ap": 0.29286,
      "num_ground_truth": 1028,
      "num_predictions": 24683,
      "true_positives": 917,
      "false_positives": 23766,
      "recall": 0.892023,
      "precision": 0.037151
    },
    "cat": {
      "ap": 0.513529,
      "num_ground_truth": 833,
      "num_predictions": 19753,
      "true_positive

In [10]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/content/O_D')
out_zip = Path('/content/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /content/O_D && zip -r /content/train.zip . -x 'public/*' 'public/**'
Created: /content/train.zip


In [17]:
from google.colab import files
files.download("/content/train.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>